# Fire distribution across Australia

## Accessing Wildfire Data via API

In [1]:
# import necessary libraries
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt


In [16]:
# 1.
# access api url

## satellite: VIIRS SNPP NRT 
## area: 'world' = entire world 
## day range: '5' = data of the last 5 days (more doesnt wooooooork)
## date: None = most recent available data, so today's data

MAP_KEY = '4899a992545cbeb46f9fd0b6a025ef17'
area_url ='https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_SNPP_NRT/world/5' # warum gehen 1, 3 oder 5 tage aber ab 8 oder so nicht mehr??

# 2.
# read in the data from URL

df_area = pd.read_csv(area_url)

# 3.
# have a first glimpse at the data

df_area.head(5)
df_area.shape

(137478, 14)

## Cleaning and Rearranging Data

### Filter for Data only within Australia

In [17]:
# define a bounding box that contains only the area of Australia based on its WGS84 coordinates

coords = [112, -44, 154, -9]

df_aus = df_area[(df_area['longitude'] >= coords[0]) & (df_area['latitude'] >= coords[1]) & (df_area['longitude'] <= coords[2]) & (df_area['latitude'] <= coords[3])].copy()
df_aus.shape
df_aus.head(20)
df_aus.tail()

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
133719,-17.37348,122.43333,334.19,0.64,0.72,2026-05-05,627,N,VIIRS,n,2.0NRT,296.54,10.72,D
133720,-17.37230,122.43025,339.55,0.64,0.72,2026-05-05,627,N,VIIRS,n,2.0NRT,296.44,6.73,D
133721,-17.37093,122.43630,332.40,0.65,0.73,2026-05-05,627,N,VIIRS,n,2.0NRT,293.85,6.73,D
133722,-17.36676,122.43181,367.00,0.64,0.72,2026-05-05,627,N,VIIRS,h,2.0NRT,295.21,10.72,D
133723,-17.36557,122.42878,353.12,0.64,0.72,2026-05-05,627,N,VIIRS,n,2.0NRT,294.93,12.76,D


### Filter? for required Timeframe or add Datetime Column with active time 

In [18]:
# 1. 
# combine the acq_date and acq_time column to one acq_datetime column and set it to an active time format with pandas function to_datetime

## acq_date is a string in the format YYYY-MM_DD, 
## while acq_time is an integer in Greenwich Mean Time (e.g. 603 meaning 6:03), 
## so it needs to be converted to string too (with astype(str)),
## fill it up to 4 numbers with zeros, so that all times have the same length (with str.zfill(4), e.g. 603 -> 0603)
## and save it as the format '%Y-%m-%d %H%M'

###df_aus['acq_datetime'] = pd.to_datetime(df_aus['acq_date'] + ' ' + df_aus['acq_time'].astype(str).str.zfill(4), format='%Y-%m-%d %H%M')
###df_aus.head()

###print (f'Australia GMT timezone datetime value range: {df_aus['acq_datetime'].min()} to {df_aus['acq_datetime'].max()}')

# 2.
# convert GMT into local time?

# 3.
# # Set the timestamp column as the index ?
###hourly_data = hourly_data.set_index("timestamp")

# Notice how 'timestamp' drops down a level to become the index!
###display(hourly_data.head(3))



### Converting raw coordinates into geometries

In [19]:
# the projection EPSG:9473 is used for Australia, as it is recommended for national mapping

# convert latitude, longitude values into point geometry and set crs (since no crs extisting yet) with crs="EPSG:9473" to EPSG:9473

gdf_aus = gpd.GeoDataFrame(
    df_aus, geometry=gpd.points_from_xy(df_aus.longitude, df_aus.latitude), crs="EPSG:9473")
print(gdf_aus.crs)
gdf_aus.head()

EPSG:9473


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,geometry
2661,-43.17316,146.83852,334.95,0.39,0.36,2026-05-01,414,N,VIIRS,n,2.0NRT,292.83,11.08,D,POINT (146.839 -43.173)
2662,-42.89930,147.85312,330.40,0.38,0.36,2026-05-01,414,N,VIIRS,n,2.0NRT,295.34,3.14,D,POINT (147.853 -42.899)
2663,-42.89856,147.85785,356.43,0.38,0.36,2026-05-01,414,N,VIIRS,n,2.0NRT,299.17,7.22,D,POINT (147.858 -42.899)
2664,-42.89783,147.86258,328.09,0.38,0.36,2026-05-01,414,N,VIIRS,n,2.0NRT,296.08,7.22,D,POINT (147.863 -42.898)
2665,-42.87638,147.86635,334.54,0.38,0.36,2026-05-01,414,N,VIIRS,n,2.0NRT,290.85,9.12,D,POINT (147.866 -42.876)


## Group geometries by date with dissolve
This creates MultiPoint Geometries that we can later add to the map that we have one layer for each day

In [20]:
gdf_daily = gdf_aus.dissolve(by="acq_date")
gdf_daily

,geometry,latitude,longitude,bright_ti4,scan,track,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
acq_date,,,,,,,,,,,,,,
2026-05-01,"MULTIPOINT (114.542 -28.105, 114.547 -28.1, 11...",-43.17316,146.83852,334.95,0.39,0.36,414,N,VIIRS,n,2.0NRT,292.83,11.08,D
2026-05-02,"MULTIPOINT (114.674 -28.353, 114.698 -28.837, ...",-43.17596,146.79762,333.68,0.53,0.42,355,N,VIIRS,n,2.0NRT,277.32,4.22,D
2026-05-03,"MULTIPOINT (114.628 -28.153, 114.628 -28.155, ...",-37.90232,142.86154,345.11,0.43,0.62,338,N,VIIRS,n,2.0NRT,287.45,4.77,D
2026-05-04,"MULTIPOINT (114.167 -27.719, 114.17 -27.722, 1...",-41.80918,147.28889,351.19,0.47,0.64,317,N,VIIRS,n,2.0NRT,283.68,6.88,D
2026-05-05,"MULTIPOINT (114.122 -27.811, 114.139 -27.848, ...",-42.05627,147.78288,334.52,0.77,0.77,258,N,VIIRS,n,2.0NRT,284.28,7.22,D


## Visualise it and create interactive Map

In [21]:
import folium

# create basemap for the extent of Australia
aus_map = folium.Map(
    location=[-25.5649, 133.1234], # use the coordinates of Australia's centre (25°56′49.3″S, 133°12′34.7″E) for the location
    zoom_start=4,
    tiles="CartoDB Positron",  # clean, light basemap
)
# add layers for each day
folium.GeoJson(gdf_daily).add_to(aus_map)

for date, row in gdf_daily.iterrows():
    fg = folium.FeatureGroup(name=str(date))
    
    folium.GeoJson(row.geometry).add_to(fg)
    
    fg.add_to(aus_map)

folium.LayerControl().add_to(aus_map)

## wie kann ich machen dass alte brände wieder verschwinden, karte addiert derzeit alle brände, bzw woher weiss ich ob/wann feuer wieder weg sind
# display the map 
aus_map.save("daily_fires_map.html")

In [9]:
# alternative way: ?
##%pip install geodatasets cartopy
##from cartopy import crs as ccrs
##from geodatasets import get_path

##path = get_path("naturalearth.land")
##world = gpd.read_file(path)

##ax = world.plot(figsize=(10, 10), color="grey", edgecolor="black")
##ax.set_xlim([coords[0],  coords[2]])
##ax.set_ylim([coords[1],  coords[3]])